In [1]:
# load templates
from cobra import Metabolite, Reaction
from cobra.io import load_json_model, save_json_model
from modelseedpy.core.mstemplate import MSTemplateBuilder
from json import load
with open("../../../ModelSEEDTemplates/templates/v6.0/Core-V5.2.json") as fh:
    template_core = MSTemplateBuilder.from_dict(load(fh)).build()
with open("../../../ModelSEEDTemplates/templates/v6.0/GramNegModelTemplateV6.json") as fh:
    template_gramneg = MSTemplateBuilder.from_dict(load(fh)).build()
with open("../../../ModelSEEDTemplates/templates/v6.0/GramPosModelTemplateV6.json") as fh:
    template_grampos = MSTemplateBuilder.from_dict(load(fh)).build()

from modelseedpy import MSMedia
media = MSMedia.from_dict({'cpd00067': 1000.0,
 'cpd00058': 1000.0,
 'cpd00013': 1000.0,
 'cpd00244': 1000.0,
 'cpd00205': 1000.0,
 'cpd00034': 1000.0,
 'cpd11574': 1000.0,
 'cpd00971': 1000.0,
#  'cpd00540': 1000.0,   # betaine
 'cpd00048': 1000.0,
 'cpd00030': 1000.0,
 'cpd00305': 100.0,
 'cpd00001': 1000.0,
 'cpd10516': 1000.0,
 'cpd00007': 1000.0,
 'cpd00159': 100.0,
 'cpd25960': 1000.0,
#  'cpd00027': 10.0,
 "cpd00009": 100,
 'cpd00063': 1000.0,
 'cpd00149': 1000.0,
 'cpd00254': 1000.0,
 'cpd00099': 1000.0})

# load default medias
from modelseedpy.core.msatpcorrection import load_default_medias
default_medias = load_default_medias()
print(f'loaded {len(default_medias)} medias')

import math
def integrate_to_model_medium(mda, model, prefix='EX_'):
    medium = {}
    for cpd, (lb, ub) in mda.get_media_constraints().items():
        rxn_exchange = f'{prefix}{cpd}'
        if rxn_exchange in model.reactions:
            medium[rxn_exchange] = math.fabs(lb)
        else:
            print('not in model', cpd)
    return medium

# load models and correct ATP
from modelseedpy import MSATPCorrection
from modelseedpy import MSGapfill
betaine_models_paths = {                                                                                                                                          
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.9.contigs__.RAST.json": "gram-pos",                                                         
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.47.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.51.contigs__.RAST.json": "gram-pos",
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.27.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.45.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.48.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.50.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_concoct_out.59.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.58.contigs__.RAST.json": "gram-neg",                                                            
      "models/Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.73.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_maxbin.047.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_C_D2_MG_DASTool_bins_concoct_out.85.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_C_D2_MG_DASTool_bins_metabat.40.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.98.contigs__.RAST.json": "gram-neg",                                                        
      "models/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.52.contigs__.RAST.json": "gram-pos",                                                            
  }

from os import path

def gapfill_model(model_path, template_label=None):
    if template_label is None:
        model_path, template_label = model_path
    template = template_gramneg if "neg" in template_label else template_grampos
    ## load model
    model = load_json_model(model_path)
    ## correct ATP
    atp_correction = MSATPCorrection(model, template_core, default_medias,
                                     compartment='c0', atp_hydrolysis_id='ATPM_c0', 
                                     load_default_medias=False)
    media_eval = atp_correction.evaluate_growth_media()
    atp_correction.determine_growth_media()
    atp_correction.apply_growth_media_gapfilling()
    atp_correction.expand_model_to_genome_scale()
    tests = atp_correction.build_tests()

    ## gapfill the model
    gapfill = MSGapfill(model, default_gapfill_templates=[template],
                        test_conditions=tests, default_target='bio1')
    gapfill_res = gapfill.run_gapfilling(media)
    gapfill.integrate_gapfill_solution(gapfill_res)
    print(f"Reactions in model after integration: {len(model.reactions)}")

    ## add missing exchange reactions for media compounds                                                                                                   
    for cpd, (lb, ub) in media.get_media_constraints().items():
        ex_id = f"EX_{cpd}"                                                                                                                                 
        if ex_id in model.reactions:  continue
        if cpd not in model.metabolites:                                                                                                                
            model.add_metabolites([Metabolite(cpd, compartment="e0")])                                                                                  
        ex_rxn = Reaction(ex_id)
        ex_rxn.lower_bound = -1000                                                                                                                      
        ex_rxn.upper_bound = 1000                     
        ex_rxn.add_metabolites({model.metabolites.get_by_id(cpd): -1})                                                                                  
        model.add_reactions([ex_rxn])
        print(f"  Added missing exchange: {ex_id}")    
   
    ## add gapfilling to the model's medium
    model.medium = integrate_to_model_medium(media, model)
    model.objective = 'bio1'
    display(model.summary())
    obj_val = model.slim_optimize()
    if obj_val > 0:
        print(f"Biomass flux: {obj_val}")
        save_json_model(model, model_path.replace('.json', '_gapfilled.json'))
    else:
        print(model_path, "failed to grow")

# for model_path, template_label in betaine_models_paths.items():
#     if path.exists(model_path.replace('.json', '_gapfilled.json')):
#         print(f"Model {model_path} already gapfilled")
#         continue
#     template = template_gramneg if "neg" in template_label else template_grampos
#     ## load model
#     model = load_json_model(model_path)
#     ## correct ATP
#     atp_correction = MSATPCorrection(model, template_core, default_medias,
#                                      compartment='c0', atp_hydrolysis_id='ATPM_c0', 
#                                      load_default_medias=False)
#     media_eval = atp_correction.evaluate_growth_media()
#     atp_correction.determine_growth_media()
#     atp_correction.apply_growth_media_gapfilling()
#     atp_correction.expand_model_to_genome_scale()
#     tests = atp_correction.build_tests()
#     new_tests = [t for t in tests if all(["." not in t['media'].id, "Glc" in t['media'].id])]
#     # for t in tests:
#     #     if "." not in t['media'].id:
#     #         new_tests.append(t)


#     ## gapfill the model
#     gapfill = MSGapfill(model, default_gapfill_templates=[template],
#                         test_conditions=new_tests, default_target='bio1')
#     gapfill_res = gapfill.run_gapfilling(media)
#     if gapfill_res is None:
#         print(f"Gapfilling failed for {model_path}")
#         gapfill = MSGapfill(model, default_gapfill_templates=[template],
#                         test_conditions=[], default_target='bio1')
#         gapfill_res = gapfill.run_gapfilling(media)
#     gapfill.integrate_gapfill_solution(gapfill_res)
#     print(f"Reactions in model after integration: {len(model.reactions)}")
   
#     ## add gapfilling to the model's medium
#     model.medium = integrate_to_model_medium(media, model)
#     model.objective = 'bio1'
#     display(model.summary())
#     obj_val = model.slim_optimize()
#     if obj_val > 0:
#         print(f"Biomass flux: {obj_val}")
#         save_json_model(model, model_path.replace('.json', '_gapfilled.json'))
#     else:
#         print(model_path, "failed to grow")

modelseedpy 0.4.3
loaded 54 medias


In [ ]:
args = [(model_path, template_label) for model_path, template_label in betaine_models_paths.items()#]
         if not path.exists(model_path.replace('.json', '_gapfilled.json'))]
print(len(args))
parallelize = True
if parallelize:
    from datetime import datetime
    from multiprocess import Pool
    from os import cpu_count

    pool_size = int(cpu_count() - 2)
    print(f"Loading {pool_size} workers and computing the scores", datetime.now())
    pool = Pool(int(pool_size))  # .map(calculate_scores, [{k: v} for k,v in pairs.items()])
    output = pool.map(gapfill_model, args)
else:
    for model_path, template_label in args:
        gapfill_model(model_path, template_label)

1
Loading 6 workers and computing the scores 2026-04-04 02:01:54.081527


No gapfilling solution found before filtering for Etho activating rxn00062_c0
No gapfilling solution found before filtering for mal-L activating rxn00062_c0
No gapfilling solution found before filtering for Pyr.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for H2.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for empty activating rxn00062_c0
No gapfilling solution found before filtering for Light activating rxn00062_c0
No gapfilling solution found before filtering for ANME activating rxn00062_c0
No gapfilling solution found before filtering for Methane activating rxn00062_c0


Reactions in model after integration: 940
  Added missing exchange: EX_cpd25960_e0


Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00007_e0,EX_cpd00007_e0,0.9751,0,0.00%
cpd00009_e0,EX_cpd00009_e0,0.5127,0,0.00%
cpd00013_e0,EX_cpd00013_e0,3.204,0,0.00%
cpd00030_e0,EX_cpd00030_e0,0.003359,0,0.00%
cpd00034_e0,EX_cpd00034_e0,0.003359,0,0.00%
cpd00048_e0,EX_cpd00048_e0,84.79,0,0.00%
cpd00058_e0,EX_cpd00058_e0,0.003359,0,0.00%
cpd00063_e0,EX_cpd00063_e0,0.003359,0,0.00%
cpd00067_e0,EX_cpd00067_e0,279.8,0,0.00%
cpd00099_e0,EX_cpd00099_e0,0.003359,0,0.00%


Biomass flux: 0.4400622894003925


Process ForkPoolWorker-2:
Process ForkPoolWorker-3:
Process ForkPoolWorker-5:
Process ForkPoolWorker-4:
Process ForkPoolWorker-6:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/andrewfreiburger/Documents/venv_official/lib/python3.10/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/andrewfreiburger/Documents/venv_official/lib/python3.10/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/andrewfreiburger/Documents/venv_official/lib/python3.10/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/andrewfreiburger/Documents/venv_official/lib/python3.10/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/andrewfreiburger/Documents/venv_official/lib/python3.10/site-packages/multiproce

In [18]:
from cobra import Reaction, Metabolite, flux_analysis
from cobra.io import load_json_model, save_json_model
from glob import glob

def add_betaine_reductase_pathway(model):
    """Add betaine reductive cleavage (Stickland) reactions to a model."""
    added = []

    # Ensure all required metabolites exist
    required_mets = {
        'cpd00540_c0': ('Glycine betaine', 'c0', 'C5H11NO2'),
        'cpd00441_c0': ('Trimethylamine', 'c0', 'C3H9N'),
        'cpd00196_c0': ('Acetyl phosphate', 'c0', 'C2H3O5P'),
        'cpd28060_c0': ('Reduced thioredoxin', 'c0', None),
        'cpd27735_c0': ('Oxidized thioredoxin', 'c0', None),
        'cpd00009_c0': ('Phosphate', 'c0', 'HO4P'),
        'cpd00067_c0': ('H+', 'c0', 'H'),
        'cpd00001_c0': ('H2O', 'c0', 'H2O'),
        'cpd00033_c0': ('Glycine', 'c0', 'C2H5NO2'),
        'cpd00013_c0': ('NH3', 'c0', 'H4N'),
        'cpd00441_e0': ('Trimethylamine', 'e0', 'C3H9N')
    }
    for met_id, (name, comp, formula) in required_mets.items():
        if met_id in model.metabolites: continue
        met = Metabolite(met_id, name=name, compartment=comp)
        if formula:  met.formula = formula
        model.add_metabolites([met])

    # rxn17220: Betaine reductase (EC 1.21.4.4)
    # cpd00540 + cpd28060 + cpd00009 + 2 cpd00067 → cpd00441 + cpd00196 + cpd27735 + cpd00001
    if 'rxn17220_c0' not in model.reactions:
        rxn = Reaction('rxn17220_c0')
        rxn.name = 'Betaine reductase'
        rxn.lower_bound = 0
        rxn.upper_bound = 1000
        rxn.add_metabolites({
            model.metabolites.get_by_id('cpd00540_c0'): -1,  # Glycine betaine
            model.metabolites.get_by_id('cpd11421_c0'): -1,  # Reduced thioredoxin
            model.metabolites.get_by_id('cpd00009_c0'): -1,  # Phosphate
            model.metabolites.get_by_id('cpd00067_c0'): 0,  # H+
            model.metabolites.get_by_id('cpd00441_c0'):  1,  # TMA
            model.metabolites.get_by_id('cpd00196_c0'):  1,  # Acetyl phosphate
            model.metabolites.get_by_id('cpd11420_c0'):  1,  # Oxidized thioredoxin
            model.metabolites.get_by_id('cpd00001_c0'):  1,  # H2O
        })
        # Auto-balance H+
        imbalance = rxn.check_mass_balance()
        if imbalance:
            h_needed = -imbalance.get('H', 0)
            # charge_needed = -imbalance.get('charge', 0)
            if h_needed != 0:
                rxn.add_metabolites({model.metabolites.get_by_id('cpd00067_c0'): h_needed})
        print(rxn.check_mass_balance())
        model.add_reactions([rxn])
        added.append('rxn17220_c0')

    # rxn07207: Glycine reductase (EC 1.21.4.2)
    # cpd00033 + cpd28060 + cpd00009 + 2 cpd00067 → cpd00196 + cpd00013 + cpd27735
    if 'rxn07207_c0' not in model.reactions:
        rxn = Reaction('rxn07207_c0')
        rxn.name = 'Glycine reductase'
        rxn.lower_bound = 0
        rxn.upper_bound = 1000
        rxn.add_metabolites({
            model.metabolites.get_by_id('cpd00033_c0'): -1,  # Glycine
            model.metabolites.get_by_id('cpd11421_c0'): -1,  # Reduced thioredoxin
            model.metabolites.get_by_id('cpd00009_c0'): -1,  # Phosphate
            model.metabolites.get_by_id('cpd00067_c0'): -1,  # H+
            model.metabolites.get_by_id('cpd00196_c0'):  1,  # Acetyl phosphate
            model.metabolites.get_by_id('cpd00013_c0'):  1,  # NH3
            model.metabolites.get_by_id('cpd11420_c0'):  1,  # Oxidized thioredoxin
            model.metabolites.get_by_id('cpd00001_c0'):  1,  # H2O
        })
        print(rxn.check_mass_balance())
        model.add_reactions([rxn])
        added.append('rxn07207_c0')

    # TMA exchange and transport
    if 'EX_cpd00441_e0' not in model.reactions:
        ex = Reaction('EX_cpd00441_e0')
        ex.name = 'TMA exchange'
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        ex.add_metabolites({model.metabolites.get_by_id('cpd00441_e0'): -1})
        model.add_reactions([ex])
        added.append('EX_cpd00441_e0')

    # Simple TMA transport (c0 <-> e0)
    if not any(r for r in model.reactions if 'cpd00441_c0' in [m.id for m in r.metabolites]
                and 'cpd00441_e0' in [m.id for m in r.metabolites]):
        tp = Reaction('rxn_TMA_transport_c0')
        tp.name = 'TMA transport'
        tp.lower_bound = -1000
        tp.upper_bound = 1000
        tp.add_metabolites({
            model.metabolites.get_by_id('cpd00441_c0'): -1,
            model.metabolites.get_by_id('cpd00441_e0'):  1,
        })
        model.add_reactions([tp])
        added.append('rxn_TMA_transport_c0')

    return added

# Apply to all gapfilled modelsb
for model_path in betaine_models_paths:
    short = model_path.split('bins')[1].split('.contigs')[0]
    print(short)
    new_path = model_path.replace(".json", "_gapfilled.json")
    model = load_json_model(new_path)
    added = add_betaine_reductase_pathway(model)

    # Test betaine reductase flux
    with model:                
        for ex in model.exchanges:
            ex.upper_bound = 1000                                                                                                                                 
        model.medium = {**model.medium, 'EX_cpd00540_e0': 10000, "EX_cpd00159_e0": 10}
        model.objective = 'bio1'
        model.reactions.get_by_id('rxn17220_c0').lower_bound = 0.01  # force minimum betaine use
        sol = flux_analysis.pfba(model)                                                                                                                       
        bet_flux = sol.fluxes.get('rxn17220_c0', 0)
        bio_flux = sol.fluxes.get('bio1', 0)                                                                                                                  
        print(f"{short}: added {added}, biomass={bio_flux:.4f}, betaine reductase={bet_flux:.4f}")    

    save_json_model(model, new_path.replace(".json", "_betaine.json"))

.concoct_out.9
{}
{}
.concoct_out.9: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=0.3352, betaine reductase=55.4443
.metabat.47
{}
{}
.metabat.47: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=0.0062, betaine reductase=0.0100
.metabat.51
{}
{}
.metabat.51: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=0.1379, betaine reductase=0.0100
_metabat.27
{}
{}
_metabat.27: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=2.0921, betaine reductase=366.8388
_metabat.45
{}
{}
_metabat.45: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=0.1328, betaine reductase=0.4770
_metabat.48
{}
{}
_metabat.48: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd00441_e0', 'rxn_TMA_transport_c0'], biomass=0.4611, betaine reductase=75.1251
_metabat.50
{}
{}
_metabat.50: added ['rxn17220_c0', 'rxn07207_c0', 'EX_cpd004

In [19]:
for model_path in betaine_models_paths:
    if "_metabat.27" not in model_path:   continue
    mdl_path = model_path.replace(".json", "_gapfilled_betaine.json")
    model = load_json_model(mdl_path)
    break

In [20]:
model.medium

{'EX_cpd00067_e0': 1000.0,
 'EX_cpd00971_e0': 1000.0,
 'EX_cpd00058_e0': 1000.0,
 'EX_cpd00013_e0': 1000.0,
 'EX_cpd00205_e0': 1000.0,
 'EX_cpd00009_e0': 100.0,
 'EX_cpd00034_e0': 1000.0,
 'EX_cpd00254_e0': 1000.0,
 'EX_cpd00149_e0': 1000.0,
 'EX_cpd00001_e0': 1000.0,
 'EX_cpd10516_e0': 1000.0,
 'EX_cpd00007_e0': 1000.0,
 'EX_cpd00305_e0': 100.0,
 'EX_cpd00030_e0': 1000.0,
 'EX_cpd00159_e0': 100.0,
 'EX_cpd00048_e0': 1000.0,
 'EX_cpd00063_e0': 1000.0,
 'EX_cpd00099_e0': 1000.0,
 'EX_cpd00244_e0': 1000.0,
 'EX_cpd11574_e0': 1000.0,
 'EX_cpd25960_e0': 1000.0,
 'EX_cpd00441_e0': 1000.0}

In [22]:
model.medium = {'EX_cpd00067_e0': 1000.0,
'EX_cpd00540_e0': 1,
 'EX_cpd00971_e0': 1000.0,
 'EX_cpd00058_e0': 1000.0,
 'EX_cpd00013_e0': 1000.0,
 'EX_cpd00244_e0': 1000.0,
 'EX_cpd00205_e0': 1000.0,
 'EX_cpd00009_e0': 100.0,
 'EX_cpd00034_e0': 1000.0,
 'EX_cpd00254_e0': 1000.0,
 'EX_cpd00048_e0': 1000.0,
 'EX_cpd00149_e0': 1000.0,
 'EX_cpd00001_e0': 1000.0,
 'EX_cpd10516_e0': 1000.0,
 'EX_cpd00305_e0': 100.0,
 'EX_cpd00030_e0': 1000.0,
 'EX_cpd00063_e0': 1000.0,
 'EX_cpd00099_e0': 1000.0,
 'EX_cpd00007_e0': 1000.0,
 'EX_cpd11574_e0': 1000.0,
 'EX_cpd00159_e0': 0.0,
 'EX_cpd25960_e0': 1000.0,
 'EX_cpd00441_e0': 1000.0}
model.objective = 'rxn00062_c0'
sol = flux_analysis.pfba(model)

model.summary(sol)

Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00001_e0,EX_cpd00001_e0,0.5,0,0.00%
cpd00540_e0,EX_cpd00540_e0,1,5,100.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00011_e0,EX_cpd00011_e0,-0.5,1,10.00%
cpd00029_e0,EX_cpd00029_e0,-0.75,2,30.00%
cpd00067_e0,EX_cpd00067_e0,-0.75,0,0.00%
cpd00441_e0,EX_cpd00441_e0,-1,3,60.00%


In [23]:
model.reactions.rxn11676_c0

Reaction identifier,rxn11676_c0
Name,acetyl-phosphate ammonia:thioredoxin disulfide oxidoreductase(glycine-forming) [c0]
Memory address,0x3352a2e60
Stoichiometry,cpd00001_c0 + cpd00013_c0 + cpd00196_c0 + cpd11420_c0 --> cpd00009_c0 + cpd00033_c0 + cpd00067_c0 + cpd11421_c0 H2O [c0] + NH3 [c0] + Acetylphosphate [c0] + trdox [c0] --> Phosphate [c0] + Glycine [c0] + H+ [c0] + trdrd [c0]
GPR,Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.27.contigs__.RAST.CDS.2141 and...
Lower bound,0.0
Upper bound,1000.0


In [24]:
model.reactions.rxn17220_c0

Reaction identifier,rxn17220_c0
Name,Betaine reductase
Memory address,0x169aaa920
Stoichiometry,cpd00009_c0 + cpd00540_c0 + cpd11421_c0 --> cpd00001_c0 + cpd00196_c0 + cpd00441_c0 + cpd11420_c0 Phosphate [c0] + BET [c0] + trdrd [c0] --> H2O [c0] + Acetylphosphate [c0] + Trimethylamine + trdox [c0]
GPR,
Lower bound,0.0
Upper bound,1000.0


In [25]:
import math
for rxn in model.reactions:
    v = math.fabs(sol.fluxes[rxn.id])
    if v > 1e-6 and 'EX_' not in rxn.id:
        print(sol.fluxes[rxn.id], rxn.id, rxn.build_reaction_string(True))
        #print(rxn.check_mass_balance())

-0.7499999999999997 rxn00225_c0 ATP [c0] + Acetate [c0] <=> ADP [c0] + Acetylphosphate [c0]
0.25000000000000006 rxn00191_c0 2-Oxoglutarate [c0] + L-Alanine [c0] <=> Pyruvate [c0] + L-Glutamate [c0]
0.2500000000000001 rxn11676_c0 H2O [c0] + NH3 [c0] + Acetylphosphate [c0] + trdox [c0] --> Phosphate [c0] + Glycine [c0] + H+ [c0] + trdrd [c0]
0.24999999999999997 rxn00931_c0 NADP [c0] + L-Proline [c0] <=> NADPH [c0] + 2 H+ [c0] + 1-Pyrroline-5-carboxylate [c0]
0.25000000000000006 rxn00184_c0 H2O [c0] + NADP [c0] + L-Glutamate [c0] <=> NADPH [c0] + NH3 [c0] + 2-Oxoglutarate [c0] + H+ [c0]
1.0 rxn05579_c0 H+ [e0] + BET [e0] <=> H+ [c0] + BET [c0]
0.25000000000000006 rxn00161_c0 NADP [c0] + L-Malate [c0] --> NADPH [c0] + CO2 [c0] + Pyruvate [c0]
-0.24999999999999994 rxn00929_c0 NAD [c0] + L-Proline [c0] <=> NADH [c0] + 2 H+ [c0] + 1-Pyrroline-5-carboxylate [c0]
0.7499999999999999 rxn05289_c0 NADPH [c0] + H+ [c0] + trdox [c0] <=> NADP [c0] + trdrd [c0]
0.25000000000000006 rxn00154_c0 NAD [c0] 

In [26]:
import escher
b = escher.Builder(map_name=f'{model.id}.energy', reaction_data=sol.fluxes.to_dict())
b.display_in_notebook()

ModuleNotFoundError: No module named 'escher'

In [ ]:
model.metabolites.cpd28060_c0.charge

0

In [ ]:
model.metabolites.cpd27735_c0.charge

1

In [ ]:
model.reactions.rxn17220_c0

Reaction identifier,rxn17220_c0
Name,Betaine reductase
Memory address,0x334e3f370
Stoichiometry,cpd00009_c0 + cpd00067_c0 + cpd00540_c0 + cpd28060_c0 --> cpd00001_c0 + cpd00196_c0 + cpd00441_c0 + cpd27735_c0 Phosphate [c0] + H+ [c0] + BET [c0] + Red-Thioredoxin [c0] --> H2O [c0] + Acetylphosphate [c0] + Trimethylamine + Ox-Thioredoxin [c0]
GPR,
Lower bound,0.0
Upper bound,1000.0


In [ ]:
for rxn in model.metabolites.cpd00540_c0.reactions:
    print(rxn.id, rxn.build_reaction_string(True))

rxn05181_c0 H2O [c0] + ATP [c0] + BET [e0] --> ADP [c0] + Phosphate [c0] + H+ [c0] + BET [c0]
rxn17220_c0 Phosphate [c0] + 2 H+ [c0] + BET [c0] + Red-Thioredoxin [c0] --> H2O [c0] + Acetylphosphate [c0] + Trimethylamine + Ox-Thioredoxin [c0]


In [ ]:
model.summary(sol) #sol

Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00001_e0,EX_cpd00001_e0,309.9,0,0.00%
cpd00007_e0,EX_cpd00007_e0,17,0,0.00%
cpd00009_e0,EX_cpd00009_e0,8.938,0,0.00%
cpd00013_e0,EX_cpd00013_e0,55.84,0,0.00%
cpd00030_e0,EX_cpd00030_e0,0.05855,0,0.00%
cpd00034_e0,EX_cpd00034_e0,0.05855,0,0.00%
cpd00048_e0,EX_cpd00048_e0,3.336,0,0.00%
cpd00058_e0,EX_cpd00058_e0,0.05855,0,0.00%
cpd00063_e0,EX_cpd00063_e0,0.05855,0,0.00%
cpd00067_e0,EX_cpd00067_e0,366,0,0.00%
